# Chapter `2.3` - Multiple Agents

### Setup and configuration

#### Importing the necessary libraries

In [ ]:
# Base utils.
from os import getenv
from dotenv import load_dotenv
import warnings

from IPython.display import Markdown

# Model init. and invocation
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool

...

Ellipsis

#### Environment settings

In [2]:
load_dotenv()

try:
    OLLAMA_MODEL = getenv("OLLAMA_MODEL", "")
    if not len(OLLAMA_MODEL):
        raise EnvironmentError("Missing Ollama model configuration in environment.")
except EnvironmentError as ee:
    print(f"ERROR: {ee}")

#### Supressing warnings

In [5]:
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Specific LangChain / LangGraph noise
warnings.filterwarnings("ignore", module="langchain")
warnings.filterwarnings("ignore", module="langgraph")

### Base tools

In [4]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number."""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number."""
    return x ** 2

### Creating subagents
- We intend to delegate tasks to *smaller subagents* in order to **distribute** the context as well as the workload of the assigned task.
- For achieving this, we have an [**orchestrator** agent](#creating-the-orchestrator-agent) in place.

In [6]:
sub_1 = create_agent(
    model=OLLAMA_MODEL,
    tools=[square_root]
)

sub_2 = create_agent(
    model=OLLAMA_MODEL,
    tools=[square]
)

### Creating subagent tools

In [19]:
@tool
def call_sub_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = sub_1.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Calculate the square root of {x} and only return the final calculated value rounded off to 3 decimal places."
                )
            ]
        }
    )
    return response["messages"][-1].content


@tool
def call_sub_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = sub_2.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Calculate the square of {x} and only return the final calculated value rounded off to 3 decimal places."
                )
            ]
        }
    )
    return response["messages"][-1].content

### Creating the orchestrator agent

In [20]:
SYS_PROMPT = "You are a helpful AI assistant who can call subagents to calculate the square root or square of a given number."

orch_agent = create_agent(
    model=OLLAMA_MODEL,
    tools=[call_sub_1, call_sub_2],
    system_prompt=SYS_PROMPT,
)

### Orchestrator agent invocation

In [26]:
question = "What is the square root of 476?"
response = orch_agent.invoke({"messages": [HumanMessage(content=question)]})

In [27]:
Markdown(response["messages"][-1].content)

The square root of 476 is approximately 21.817.